In [ ]:
!pip install -r requirements.txt

  Using cached deeplake-3.6.19.tar.gz (532 kB)
  Preparing metadata (setup.py) ... done
  Using cached openai-0.27.8-py3-none-any.whl (73 kB)
  Using cached tiktoken-0.4.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.7 MB)
  Using cached transformers-4.32.0-py3-none-any.whl (7.5 MB)
  Using cached torch-2.0.1-cp310-cp310-manylinux1_x86_64.whl (619.9 MB)
  Using cached deepspeed-0.10.1.tar.gz (851 kB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.7.1-py3-none-any.whl (117 kB)
  Using cached peft-0.5.0-py3-none-any.whl (85 kB)
  Using cached wandb-0.15.8-py3-none-any.whl (2.1 MB)
  Using cached bitsandbytes-0.41.1-py3-none-any.whl (92.6 MB)
  Using cached accelerate-0.22.0-py3-none-any.whl (251 kB)
  Using cached neural_compressor-2.2.1-py3-none-any.whl (1.3 MB)
  Using cached onnx-1.14.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (14.6 MB)
  Using cached pandas-2.0.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.3 MB)
  U

In [ ]:
!nvidia-smi

Thu Mar 14 23:21:21 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   59C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoModelForCausalLM,
                          DataCollatorWithPadding,
                          TrainingArguments,
                          Trainer)
from datasets import load_dataset, ClassLabel, DatasetDict
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch, time, os
from torch.optim.lr_scheduler import StepLR

from dotenv import load_dotenv
load_dotenv()

from accelerate import Accelerator
import evaluate
import numpy as np

[2024-03-14 22:55:26,940] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [ ]:
from huggingface_hub import login

login(
  token=os.getenv('HUGGINGFACEHUB_API_TOKEN'), # ADD YOUR TOKEN HERE
  add_to_git_credential=True
)

In [ ]:
model_id = 'facebook/opt-1.3b'
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [ ]:
print(tokenizer)

GPT2TokenizerFast(name_or_path='facebook/opt-1.3b', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'unk_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'pad_token': AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True)}, clean_up_tokenization_spaces=True)


In [ ]:
dataset_id = 'FinGPT/fingpt-sentiment-train'

dataset = load_dataset(dataset_id, split='train')

train_test_dataset = dataset.train_test_split(test_size=0.3)
test_valid_dataset = train_test_dataset['test'].train_test_split(test_size=0.5)

dataset = DatasetDict({
    'train': train_test_dataset['train'],
    'test': test_valid_dataset['test'],
    'valid': test_valid_dataset['train']})

print(f"Train dataset size: {len(dataset['train'])}")
print(f"Test dataset size: {len(dataset['test'])}")
print(f"Validation dataset size: {len(dataset['valid'])}")

Generating train split:   0%|          | 0/76772 [00:00<?, ? examples/s]

Train dataset size: 53740
Test dataset size: 11516
Validation dataset size: 11516


In [ ]:
label_names = sorted(set(dataset["train"]["output"]))
dataset = dataset.cast_column("output", ClassLabel(names=label_names))

Casting the dataset:   0%|          | 0/53740 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/11516 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/11516 [00:00<?, ? examples/s]

In [ ]:
dataset['train']

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 53740
})

In [ ]:
def formatting_prompts_func(example):
    """Prepare the text from a sample of the dataset."""
    text = f"{example['instruction']}\n\nContent: {example['input']}\n\nSentiment: {example['labels']}"
    return text

In [ ]:
def preprocess_function(samples):
    return tokenizer(samples['input'],
                     # padding="max_length", (if you need ixed length padding)
                     truncation=True,
                     max_length=128)


def tokenize_data(dataset):
    tokenized_data = dataset.map(preprocess_function,
                                 batched=True
                                 )
    tokenized_data = tokenized_data.rename_column('output', 'labels')
    tokenized_data = tokenized_data.with_format('torch')
    return tokenized_data

In [ ]:
tokenized_data = tokenize_data(dataset)

Map:   0%|          | 0/53740 [00:00<?, ? examples/s]

Map:   0%|          | 0/11516 [00:00<?, ? examples/s]

Map:   0%|          | 0/11516 [00:00<?, ? examples/s]

In [ ]:
labels = tokenized_data["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [ ]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'instruction', 'input_ids', 'attention_mask'],
        num_rows: 53740
    })
    test: Dataset({
        features: ['input', 'labels', 'instruction', 'input_ids', 'attention_mask'],
        num_rows: 11516
    })
    valid: Dataset({
        features: ['input', 'labels', 'instruction', 'input_ids', 'attention_mask'],
        num_rows: 11516
    })
})

In [ ]:
# model2 = AutoModelForSequenceClassification.from_pretrained(
#     model_id,
#     num_labels=9,
#     id2label=id2label,  # For converting predictions to strings
#     label2id=label2id,
# )

pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Some weights of OPTForSequenceClassification were not initialized from the model checkpoint at facebook/opt-1.3b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import LoraConfig

"""
 Overview of the supported task types:
    - SEQ_CLS: Text classification.
    - SEQ_2_SEQ_LM: Sequence-to-sequence language modeling.
    - Causal LM: Causal language modeling.
    - TOKEN_CLS: Token classification.
    - QUESTION_ANS: Question answering.
    - FEATURE_EXTRACTION: Feature extraction. Provides the hidden states which can be used as embeddings or features
      for downstream tasks.
"""

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["k_proj","v_proj","q_proj","out_proj"],
    task_type="SEQ_CLS",
)

In [ ]:
from transformers import Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained(model_id)
repo_id = model_id + '-lora'

# Define training args
training_args = TrainingArguments(
    output_dir= repo_id,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=5e-5,
	  num_train_epochs=10,
	  # PyTorch 2.0 specifics
    # bf16=True,
    fp16=True, # bfloat16 training
	  torch_compile=True, # optimizations
    optim="adamw_torch_fused", # improved optimizer
    # logging & evaluation strategies
    logging_dir=f"{repo_id}/logs",
    logging_strategy="steps",
    logging_steps=200,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True
    #metric_for_best_model="f1"
    # push to hub parameters
    # report_to="tensorboard"
    )

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

The speedups for torchdynamo mostly come wih GPU Ampere or higher and which is not detected here.


In [ ]:
from trl import SFTTrainer
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

data_collator = DataCollatorWithPadding(tokenizer)

# Create a Trainer instance
sfttrainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['valid'],
    peft_config=lora_config,
    data_collator=data_collator,
    formatting_func=formatting_prompts_func,
    dataset_text_field='input',
    packing=True,
    max_seq_length=512
    )

/usr/local/lib/python3.10/dist-packages/trl/trainer/utils.py:465: UserWarning: The passed formatting_func has more than one argument. Usually that function should have a single argument `example` which corresponds to the dictionary returned by each element of the dataset. Make sure you know what you are doing.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 394.00 MiB (GPU 0; 14.75 GiB total capacity; 14.58 GiB already allocated; 7.06 MiB free; 14.61 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
print(model2)

OPTForSequenceClassification(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(
              in_features=2048, out_features=2048, bias=True
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.05, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=2048, out_features=16, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=16, out_features=2048, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
            )
            (v_proj): Li

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

print( print_trainable_parameters(sfttrainer.model))

trainable params: 6328320 || all params: 1322086400 || trainable%: 0.47866160638215477
None


In [ ]:
import wandb

wandb.login(key='b9b9654b087057892024ae51610c0c59e1d05c2d')

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
torch.cuda.empty_cache()
import gc

gc.collect()



325

In [ ]:
sfttrainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 14.75 GiB total capacity; 14.58 GiB already allocated; 7.06 MiB free; 14.61 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
# from peft import PeftModel

# model = PeftModel.from_pretrained(model, f"./{repo_id}/<desired_checkpoint>/")
# model.eval()
# model = model.merge_and_unload()

# model.save_pretrained(f"./{repo_id}/merged")